# Week 3 Lab: Prompting for Effective LLM Use

Focus: clear rules, roles (system vs. user), separating data from instructions, formatting outputs for parsing, two-step prompting, and a small optional EDA demo.

Set GEMINI_MODEL to a model available to your account; record it with your results.

## 0) Setup

- pip install google-genai python-dotenv pandas numpy seaborn matplotlib scipy

Put your API key in .env as GEMINI_API_KEY=... and restart the kernel if needed.
Set GEMINI_MODEL to a model available to your account. For the EDA cell, prepare the Docker image as described below.


In [13]:
pip install google-genai python-dotenv pandas numpy seaborn matplotlib scipy

In [14]:
import google.genai
import dotenv
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy

print("Toutes les bibliothèques sont correctement importées.")
print("Google GenAI :", google.genai.__name__)
print("Pandas :", pd.__version__)
print("NumPy :", np.__version__)
print("SciPy :", scipy.__version__)

Toutes les bibliothèques sont correctement importées.
Google GenAI : google.genai
Pandas : 2.2.3
NumPy : 2.4.6
SciPy : 1.16.3


In [ ]:
from pathlib import Path

env_content = """GEMINI_API_KEY=chercher_sur google cloud 

GEMINI_MODEL=gemini-3.5-flash
"""

Path(".env").write_text(env_content)

print("Fichier .env créé avec succès !")

Fichier .env créé avec succès !


In [33]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GEMINI_MODEL = os.getenv("GEMINI_MODEL")

print("Clé API chargée :", bool(GEMINI_API_KEY))
print("Modèle sélectionné :", GEMINI_MODEL)

Clé API chargée : True
Modèle sélectionné : gemini-3.5-flash


In [36]:
from __future__ import annotations

import os, json, textwrap
from typing import Any, Dict

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

from dotenv import load_dotenv
load_dotenv()

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
assert GEMINI_API_KEY, 'Please set GEMINI_API_KEY in your environment or .env file.'

from google import genai
from google.genai import types
client = genai.Client(api_key=GEMINI_API_KEY, http_options={'timeout': 60000})
MODEL_NAME = os.environ['GEMINI_MODEL']  # choose an available model before class
GEN_CONFIG = types.GenerateContentConfig(
    max_output_tokens=800,
    thinking_config=types.ThinkingConfig(thinking_level="low"))
GEN_CONFIG_EXTENDED = types.GenerateContentConfig(
                       max_output_tokens=2000,
                      thinking_config=types.ThinkingConfig(thinking_level="low"))

class PromptModel:
    """Keep the lab's generate_content interface with the current client SDK."""
    def __init__(self, system_instruction=None):
        self.system_instruction = system_instruction

    def generate_content(self, prompt, generation_config=None):
        config = generation_config or GEN_CONFIG
        config = config.model_copy(update={'system_instruction': self.system_instruction})
        return client.models.generate_content(model=MODEL_NAME, contents=prompt, config=config)

def make_model(system_instruction=None):
    return PromptModel(system_instruction)

model_default = make_model()


## 1) Clear rules for prompting

- Be specific: ask only for what you need; set scope and length.
- State a role: use a system instruction for role and constraints.
- Separate parts: instructions, data, and output format.
- Format outputs: require strict JSON or a fenced code block.
- Acceptance criteria: add a checklist and self-verify.
- Few-shot (minimal): one short example can anchor style.

In [35]:
# --- Bad Prompt: Vague and unstructured ---
bad_prompt = "Tell me about large language models."
print("--- Bad Prompt ---")
print(model_default.generate_content(bad_prompt, generation_config=GEN_CONFIG_EXTENDED).text)


# --- Good Prompt: Clear, role-based, and structured ---
good_prompt = textwrap.dedent('''
    You are an expert technical writer.
    Explain the concept of "Large Language Models" to a university student.
    - Start with a one-sentence definition.
    - Provide three key bullet points on how they work.
    - Limit the total response to under 100 words.
''')
print("\n--- Good Prompt ---")
print(model_default.generate_content(good_prompt, generation_config=GEN_CONFIG).text)

--- Bad Prompt ---
**Large Language Models (LLMs)** are a type of artificial intelligence (AI) designed to understand, process, and generate human language. They power tools like ChatGPT, Claude, and Google Gemini, and represent one of the biggest breakthroughs in computer science in recent decades.

Here is a comprehensive overview of how they work, what they do, their limitations, and where they are headed.

---

### 1. How Do LLMs Work?

At their core, LLMs are **advanced pattern recognition machines**. They do not "think" or "feel" like humans; instead, they use complex statistics to predict what word should come next in a sentence.

*   **The "Large" Part:** 
    *   **Data:** They are trained on vast amounts of text—books, articles, websites, code, and social media posts (often hundreds of billions or trillions of words).
    *   **Parameters:** Parameters are the internal "settings" or "weights" the model uses to make decisions. Modern LLMs have tens to hundreds of billions (or 

### Exercise
Improve the following vague prompt by applying at least three of the "clear rules" (e.g., add a role, specify the format, set constraints).

**Vague prompt:** `"Why should I learn data science?"`

## 2) Role prompting changes outcomes (system vs. user)

We keep the user prompt identical and only change the system instruction.

In [37]:
PROMPT = 'In one sentence, why should I learn skateboarding?'

resp_plain = model_default.generate_content(PROMPT, generation_config=GEN_CONFIG_EXTENDED)
print('No system instruction:', resp_plain.text)

model_cat = make_model('You are a cat.')
resp_cat = model_cat.generate_content(PROMPT, generation_config=GEN_CONFIG_EXTENDED)
print('System: You are a cat.', resp_cat.text)


No system instruction: Learning to skateboard builds physical balance and mental resilience while giving you a fun, creative way to express yourself and explore your environment.
System: You are a cat. Learning to skateboard grants you feline-like balance and agility, finally allowing a human to experience the supreme joy of high-speed zoomies on four wheels, meow.


### Exercise
Using the same `PROMPT` about skateboarding, create a new model with the system instruction `"You are a worried parent."` and generate a response. How does it differ from the "cat" and default responses?

In [39]:
PROMPT = 'In one sentence, why should I learn skateboarding?'

resp_plain = model_default.generate_content(PROMPT, generation_config=GEN_CONFIG_EXTENDED)
print('No system instruction:', resp_plain.text)

model_parents = make_model('You are a worried parent')
resp_cat = model_parents.generate_content(PROMPT, generation_config=GEN_CONFIG_EXTENDED)
print('System: You are a worried parent.', resp_cat.text)


No system instruction: Learning skateboarding offers an exhilarating way to improve your physical fitness and creative self-expression, while teaching you invaluable mental resilience by empowering you to get back up every time you fall.
System: You are a worried parent. I am terrified you are going to break something, but I have to admit it teaches incredible balance, resilience, and how to get back up when life knocks you down—just please promise me you will wear every single piece of protective gear!


## 3) Separate data from instructions

Keep your instructions stable and swap data safely using delimiters.

In [40]:
instructions = textwrap.dedent('''
Summarize the data in one short sentence and produce 3 topical tags.
Return strict JSON with keys summary (str) and tags (list of str).
''').strip()
data_block = textwrap.dedent('''<data>
Skateboarding participation has risen globally over the past decade,
with growing inclusion in major events and broader demographics. During the last olympic games, skateboarding made its debut,
highlighting its increasing recognition as a competitive sport. The culture around skateboarding continues to evolve,
influencing fashion, music, and lifestyle trends worldwide.
</data>''').strip()

prompt = f'Instructions: {instructions}{data_block}'
resp = model_default.generate_content(prompt, generation_config=GEN_CONFIG)
text = resp.text
print(text)
# Strip markdown fences if the model added them
if text.strip().startswith("```json"):
    text = text.strip().removeprefix("```json\n").removesuffix("\n```")
try:
    parsed = json.loads(text)
    print('Parsed:', parsed)
except Exception as e:
    print('Parsing failed; consider asking the model to fix format.')


```json
{
  "summary": "Global skateboarding participation and demographic diversity have grown, highlighted by its Olympic debut and widespread influence on modern culture.",
  "tags": [
    "Skateboarding",
    "Sports",
    "Culture"
  ]
}
```
Parsed: {'summary': 'Global skateboarding participation and demographic diversity have grown, highlighted by its Olympic debut and widespread influence on modern culture.', 'tags': ['Skateboarding', 'Sports', 'Culture']}


### Exercise
Use the `instructions` variable from the example above, but replace the `data_block` with a new paragraph about a topic of your choice. Verify that the model still follows the instructions and produces the correct JSON output.

## 4) Formatting outputs for parsing

Ask for strict JSON and validate with json.loads. If parsing fails, ask the model to fix the format.

In [41]:
schema = 'title: str; bullets: list[str]'
prompt = (
    'System: Return strict JSON with keys title (str) and bullets (list[str]).'
    'User:'
    '- Task: Summarize why clear prompts matter for data science.'
    '- Length: Title + 3 bullets.'
    f'- Output: {schema}'
)
resp = model_default.generate_content(prompt, generation_config=GEN_CONFIG)
text = resp.text or ''
print(text)

# Strip markdown fences if the model added them
if text.strip().startswith("```json"):
    text = text.strip().removeprefix("```json\n").removesuffix("\n```")

try:
    obj = json.loads(text)
    print('Valid JSON with', len(obj.get('bullets', [])), 'bullets')
except json.JSONDecodeError as e:
    print('JSON parse error:', e)


{
  "title": "Why Clear Prompts Matter in Data Science",
  "bullets": [
    "Ensures accurate interpretation of complex analytical tasks, logic, and dataset constraints.",
    "Reduces iteration cycles by avoiding misaligned code generation, model selection, and data transformations.",
    "Enhances reproducibility and consistency across automated AI-driven data pipelines."
  ]
}
Valid JSON with 3 bullets


### Exercise
Modify the prompt to request a different JSON schema. Ask the model to return an object with two keys: `topic` (a string) and `key_takeaways` (a list of exactly two strings).

## 5) Two-step prompting (prompt-writer)

Use the model to craft the prompt first, then run that prompt.

In [42]:
task = textwrap.dedent('''
Write exactly 5 concise bullets for an EDA plan on a churn dataset
with columns: age, tenure_months, monthly_charges, contract_type, churn.
''').strip()

prompt_writer = textwrap.dedent('''
You are a prompt engineer. Draft a clear, minimal prompt for another LLM to:
- Role: senior data scientist
- Task: {TASK}
- Output: exactly 5 bullets in plain text
- Constraints: be specific, no extra commentary
Return ONLY the prompt text that I can paste into the other model.
''').format(TASK=task)

response = model_default.generate_content(prompt_writer, generation_config=GEN_CONFIG_EXTENDED)
draft = response.text if response.text else "Error: No response generated"
print('--- Drafted prompt ---')
print(draft)

if draft != "Error: No response generated":
	response = model_default.generate_content(draft, generation_config=GEN_CONFIG_EXTENDED)
	result = response.text if response.text else "Error: No response generated"
	print('\n--- Result ---')
	print(result)


--- Drafted prompt ---
You are a senior data scientist. Write an exploratory data analysis (EDA) plan for a customer churn dataset containing the following columns: `age`, `tenure_months`, `monthly_charges`, `contract_type`, and `churn`.

Constraints:
- Output exactly 5 concise bullet points.
- Be specific to the provided dataset columns and churn analysis.
- Plain text only.
- Do not include any introductory, concluding, or extra commentary.

--- Result ---
- Validate data health by checking missing values, data types, and the baseline distribution of the target variable churn, while calculating summary statistics for age, tenure_months, and monthly_charges.
- Evaluate the categorical relationship between contract_type and churn using proportion analysis and chi-square independence tests to pinpoint high-risk contract groups.
- Compare numerical feature distributions (age, tenure_months, monthly_charges) across churned versus retained customers using boxplots, KDE plots, and Mann-Whit

### Exercise
Use the two-step "prompt-writer" pattern for a different task. First, have the model generate a prompt that asks another LLM to write a four-line poem about Marseille. Then, execute the generated prompt.

## 6) Optional: quick EDA demo (code generation)

We generate a small dataset and ask the model for a minimal analyze(df) function.
We require ONLY raw Python code (no fences) to simplify execution.
Execution uses a disposable Docker container, with the dataset mounted read-only. Before class, start Docker and build the lab image from the repository root:

```sh
docker build -t llm-course-eda -f labs/Dockerfile.eda .
```

The container returns printed output and plot images. Check both against the data. Week 5 extends this generate/run exercise with error feedback and revision.


In [ ]:
rng = np.random.default_rng(42)
n = 600
age = rng.integers(18, 75, size=n)
tenure = rng.integers(1, 72, size=n)
monthly = rng.normal(65, 20, size=n).clip(5, 200)
contract_type = rng.choice(['month-to-month','one-year','two-year'], size=n, p=[0.55,0.25,0.20])
p_churn = 1/(1+np.exp(-(0.35 - 0.003*tenure - 0.10*(contract_type=='two-year') - 0.05*(contract_type=='one-year') + 0.0015*(monthly-65))))
churn = rng.binomial(1, p_churn)
df = pd.DataFrame({'age': age, 'tenure_months': tenure, 'monthly_charges': monthly, 'contract_type': contract_type, 'churn': churn})
df.head()


In [ ]:
eda_prompt = (
    'Return ONLY Python code (no fences) defining a function `def analyze(df):`. '
    'Inside the function: '
    '1) Use pandas, seaborn, and matplotlib to perform a basic EDA. '
    '2) Print the results of df.describe(). '
    '3) Create two interesting plots (e.g., a histogram and a boxplot) to visualize the data. '
    '4) Do NOT call plt.show() or plt.close() inside the function. '
    '5) The function should not return any value. '
)

response = model_default.generate_content(eda_prompt, generation_config=GEN_CONFIG_EXTENDED)

# Check if the response was blocked before proceeding
if not response.candidates:
    print("Code generation failed. The response was blocked.")
    print("Reason:", response.prompt_feedback)
else:
    try:
        eda_code = response.text
        # Strip markdown fences if the model added them
        if eda_code.strip().startswith("```python"):
            eda_code = eda_code.strip().removeprefix("```python\n").removesuffix("\n```")
        # Add a print statement to debug the generated code
        print("--- Generated Code ---")
        print(eda_code)
        print("----------------------")

        from pathlib import Path
        import sys
        course_root = Path.cwd() if (Path.cwd() / 'labs').is_dir() else Path.cwd().parent
        sys.path.insert(0, str(course_root / 'labs'))
        from python_sandbox import DockerPythonSandbox, eda_program, save_eda_output
        work = course_root / 'local/student/2026-2027/eda'
        work.mkdir(parents=True, exist_ok=True)
        csv_path = work / 'churn.csv'
        df.to_csv(csv_path, index=False)
        execution = DockerPythonSandbox(csv_path, image='llm-course-eda',
                                        output_limit=1048576).execute_python(eda_program(eda_code))
        print(execution['status'], execution['exit_code'])
        if execution['status'] == 'completed' and execution['exit_code'] == 0:
            report, images = save_eda_output(execution['output'], work)
            print(report)
            from IPython.display import Image, display
            for image in images:
                display(Image(filename=str(image)))
        else:
            print(execution['output'])  # use this feedback when revising the code
    except Exception as e:
        print(f"\nAn error occurred while executing the generated code: {e}")


### Exercice
Play with the above code generation model. How far can you push it?

### Exercice
Use the techniques of your choice to analyse the dataset `\data\weightHeightSex.txt`.  